# Phase 3: Evaluation on Colab 


In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl" peft accelerate bitsandbytes huggingface_hub
!pip install -q -U datasets!pip install -q pandas

##  Load Test Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset("myounes21/logos-reasoning-dataset", split="train")

split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
test_dataset = split_dataset['test']
print(f" Loaded {len(test_dataset)} test examples for evaluation.")

##  Evaluation Utilities


In [ ]:
import re
import time
import torch
import gc
import signal
import json as json_module

def build_prompt(instruction):
    return f"<|im_start|>user\n{instruction}<|im_end|>\n<|im_start|>assistant\n"

def generate_completion(model, tokenizer, prompt, max_new_tokens=1024):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start_time
    generated_tokens = outputs.shape[1] - inputs['input_ids'].shape[1]
    tps = generated_tokens / elapsed if elapsed > 0 else 0
    completion = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return completion, tps

def extract_code_block(text):
    matches = re.findall(r'```python\n(.*?)```', text, re.DOTALL)
    return matches[-1].strip() if matches else None

def run_unit_tests(code, unit_tests, timeout=3):
    if not code or not unit_tests:
        return None 

    func_names = [f for f in re.findall(r'^def (\w+)\(', code, re.MULTILINE)
                  if not f.startswith('__')]
    if not func_names:
        return 0.0 
    func_name = func_names[-1]

    namespace = {}
    try:
        exec(code, namespace)
    except Exception:
        return 0.0 

    func = namespace.get(func_name)
    if not callable(func):
        return 0.0

    passed = 0
    total = len(unit_tests)

    for test in unit_tests:
        inp = test.get('input', '')
        expected = test.get('expected', '')
        try:
            call_str = f'{func_name}({inp})'
            signal.alarm(timeout)
            result = eval(call_str, namespace)
            signal.alarm(0)

            if isinstance(expected, str):
                try:
                    expected_val = eval(expected)
                except:
                    expected_val = expected
            else:
                expected_val = expected

            if result == expected_val:
                passed += 1
        except Exception:
            signal.alarm(0)
            continue

    return 5.0 * (passed / total) if total > 0 else 0.0

def score_completion(completion, unit_tests=None):
    scores = {}

    has_think = "<think>" in completion and "</think>" in completion
    has_code = "```python" in completion
    scores['format'] = 1.0 if (has_think and has_code) else 0.0

    scores['has_function'] = 1.0 if "def " in completion else 0.0

    code = extract_code_block(completion)
    run_score = run_unit_tests(code, unit_tests)
    scores['code_correct'] = run_score / 5.0 if run_score is not None else 0.0

    think_match = re.search(r'<think>(.*?)</think>', completion, re.DOTALL)
    think_text = think_match.group(1) if think_match else ""
    arabic_chars = len(re.findall(r'[؀-ۿ]', think_text))
    total_chars = len(think_text.strip())
    scores['arabic_ratio'] = arabic_chars / max(total_chars, 1)

    keywords = ["إذن", "بالتالي", "لأن", "بما أن", "نستنتج", "أولاً", "ثانياً", "أخيراً"]
    scores['logic_keywords'] = sum(1 for kw in keywords if kw in think_text)

    return scores

print(" Evaluation utilities loaded.")

##  Run Evaluation


In [ ]:
from unsloth import FastLanguageModel
import pandas as pd
from IPython.display import display

MODEL_VARIANTS = {
    "Base Qwen": {
        "model_name": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        "adapter_path": None,
    },
    "SFT Only": {
        "model_name": "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        "adapter_path": "myounes21/logos-sft-adapter",
    },
    "LOGOS Full (SFT+GRPO)": {
        "model_name": "myounes21/logos-sft-adapter",
        "adapter_path": "myounes21/logos-grpo-adapter",
    },
}

MAX_EVAL_SAMPLES = len(test_dataset)
eval_subset = test_dataset.select(range(min(MAX_EVAL_SAMPLES, len(test_dataset))))

all_results = []
per_sample_results = []

for variant_name, config in MODEL_VARIANTS.items():
    print(f"\n{'='*60}")
    print(f" Evaluating: {variant_name}")
    print(f"{'='*60}")

    gc.collect()
    torch.cuda.empty_cache()

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = config["model_name"],
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )

    if config["adapter_path"]:
        try:
            model.load_adapter(config["adapter_path"])
            print(f"   Adapter loaded: {config['adapter_path']}")
        except Exception as e:
            print(f"   Could not load adapter: {e}")
            print(f"  Skipping {variant_name}...")
            continue

    FastLanguageModel.for_inference(model)

    variant_scores = {
        'format': [], 'has_function': [], 'arabic_ratio': [],
        'logic_keywords': [], 'tps': [], 'code_correct': []
    }

    for i, example in enumerate(eval_subset):
        prompt = build_prompt(example['instruction'])
        completion, tps = generate_completion(model, tokenizer, prompt)
        
        ut = example.get('unit_tests', [])
        if isinstance(ut, str):
            try:
                ut = json_module.loads(ut)
            except:
                ut = []
                
        scores = score_completion(completion, unit_tests=ut)

        for key in scores:
            variant_scores[key].append(scores[key])
        variant_scores['tps'].append(tps)
        
        sample_res = {
            'Model': variant_name,
            'Sample_ID': i,
            'Difficulty': example.get('difficulty', 'Unknown'),
            'TPS': tps
        }
        sample_res.update(scores)
        per_sample_results.append(sample_res)

        if (i + 1) % 5 == 0:
            print(f"  Progress: {i+1}/{len(eval_subset)} samples")

    result = {
        'Model': variant_name,
        'Format %': f"{sum(variant_scores['format'])/len(variant_scores['format'])*100:.0f}%",
        'Has Function %': f"{sum(variant_scores['has_function'])/len(variant_scores['has_function'])*100:.0f}%",
        'Code Correct %': f"{sum(variant_scores['code_correct'])/len(variant_scores['code_correct'])*100:.0f}%",
        'Arabic Ratio': f"{sum(variant_scores['arabic_ratio'])/len(variant_scores['arabic_ratio']):.2f}",
        'Logic Keywords (avg)': f"{sum(variant_scores['logic_keywords'])/len(variant_scores['logic_keywords']):.1f}",
        'Tokens/sec': f"{sum(variant_scores['tps'])/len(variant_scores['tps']):.1f}",
    }
    all_results.append(result)
    print(f"   Done: {result}")

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*60)
print(" All evaluations complete!")

##  Results Table

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(all_results)
print("\n Final Benchmark Results")
print("=" * 60)
display(df)

print("\n Correctness by Difficulty")
print("=" * 60)
raw_df = pd.DataFrame(per_sample_results)
for difficulty in ['سهل', 'متوسط', 'صعب']:
    subset = raw_df[raw_df['Difficulty'] == difficulty]
    if not subset.empty:
        for model_name in raw_df['Model'].unique():
            mod_sub = subset[subset['Model'] == model_name]
            if not mod_sub.empty:
                acc = mod_sub['code_correct'].mean() * 100
                print(f"Difficulty: {difficulty:<10} | Model: {model_name:<25} | Code Correct: {acc:.0f}%")

csv_path = "/content/logos_benchmark_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n Aggregate results saved to: {csv_path}")

raw_csv_path = "/content/logos_raw_results.csv"
raw_df.to_csv(raw_csv_path, index=False)
print(f" Raw per-sample results saved to: {raw_csv_path}")

##  Sample Outputs (Qualitative Comparison)


In [ ]:
sample_idx = 0
sample = test_dataset[sample_idx]
prompt = build_prompt(sample['instruction'])

print(f" Problem:\n{sample['instruction']}\n")
print(f" Ground Truth Answer:\n{sample['answer'][:500]}\n")
print("="*60)

gc.collect()
torch.cuda.empty_cache()

adapter_loaded = False
for adapter_name, base_model_name, adapter_path in [
    ("LOGOS Full", "myounes21/logos-sft-adapter", "myounes21/logos-grpo-adapter"),
    ("SFT Only", "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit", "myounes21/logos-sft-adapter"),
]:
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name = base_model_name,
            max_seq_length = 2048,
            dtype = None,
            load_in_4bit = True,
        )
        model.load_adapter(adapter_path)
        print(f"Using: {adapter_name}")
        adapter_loaded = True
        break
    except Exception as e:
        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()
        continue

if not adapter_loaded:
    print("Using: Base Qwen (no adapter found)")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )

FastLanguageModel.for_inference(model)
completion, tps = generate_completion(model, tokenizer, prompt)

print(f"\n Model Output ({tps:.1f} tok/s):\n")
print(completion[:1500])